# WorldSim DGX Spark QLoRA Baseline Training

This notebook is the hardened DGX baseline path. It uses the shared baseline trainer and fails early if the runtime cannot satisfy true CUDA QLoRA.


## 1. Repo Root Guard


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
repo_marker = cwd / 'training/run_qlora_train.py'
notebook_marker = cwd / 'notebooks/dgx_spark_qlora_train.ipynb'
assert repo_marker.exists() and notebook_marker.exists(), (
    f'Run this notebook from the WorldSim repo root. cwd={cwd} repo_marker={repo_marker}'
)
{
    'python_executable': sys.executable,
    'cwd': str(cwd),
    'repo_marker': str(repo_marker),
    'repo_marker_exists': repo_marker.exists(),
}


## 2. Environment Visibility


In [ ]:
from training.lib.qlora_smoke import get_environment_summary

environment = get_environment_summary()
torch_info = environment.get('torch', {})
env_visibility = {
    'python_executable': environment.get('python', {}).get('executable'),
    'cwd': environment.get('cwd'),
    'torch_version': torch_info.get('version'),
    'torch_cuda_version': torch_info.get('cuda_version'),
    'torch_cuda_available': torch_info.get('cuda_available'),
    'gpu_count': torch_info.get('cuda_device_count', 0),
    'gpu_names': torch_info.get('cuda_device_names', []),
    'bitsandbytes_available': environment.get('bitsandbytes', {}).get('available'),
    'bitsandbytes_version': environment.get('bitsandbytes', {}).get('version'),
    'environment': environment,
}
env_visibility


## 3. Strict True-QLoRA Preflight


In [ ]:
from training.lib.qlora_smoke import get_true_qlora_preflight

preflight = get_true_qlora_preflight()
preflight
assert preflight['ok'], preflight['blocker_reason']


## 4. Baseline Config


In [ ]:
from datetime import UTC, datetime
from pathlib import Path

from training.lib.qlora_smoke import BASELINE_MODEL_NAME, resolve_baseline_notebook_config

RUN_ID = datetime.now(UTC).strftime('run-%Y%m%dT%H%M%SZ')
CONFIG = resolve_baseline_notebook_config(
    RUN_ID,
    overrides={
        'model_name': BASELINE_MODEL_NAME,
        'dry_run': False,
        'require_qlora': True,
        'max_train_samples': 0,
        'max_eval_samples': 0,
        'max_steps': 200,
        'gradient_accumulation_steps': 8,
        'learning_rate': 1e-4,
        'logging_steps': 5,
        'eval_steps': 25,
        'save_steps': 25,
        'save_total_limit': 2,
    },
)
CONFIG['output_dir'] = Path('outputs/baseline/worldsim-v31-mix-v1') / CONFIG['run_id']
CONFIG


## 5. Dataset Preview


In [ ]:
from scripts.common import read_jsonl

train_preview_rows = read_jsonl(CONFIG['train_file'])[:3]
dataset_preview = [
    {
        'task': row.get('task'),
        'system': row['messages'][0]['content'],
        'user': row['messages'][1]['content'],
        'assistant': row['messages'][-1]['content'],
    }
    for row in train_preview_rows
]
dataset_preview


## 6. Trainer Invocation


In [ ]:
import time

from training.lib.qlora_smoke import SmokeRunConfig, run_baseline_or_raise

cfg = SmokeRunConfig(
    run_mode='baseline',
    model_name=CONFIG['model_name'],
    train_file=CONFIG['train_file'],
    dev_file=CONFIG['dev_file'],
    output_dir=CONFIG['output_dir'],
    max_steps=CONFIG['max_steps'],
    max_train_samples=CONFIG['max_train_samples'],
    max_eval_samples=CONFIG['max_eval_samples'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    logging_steps=CONFIG['logging_steps'],
    eval_steps=CONFIG['eval_steps'],
    save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    require_qlora=CONFIG['require_qlora'],
    seed=CONFIG['seed'],
    dry_run=CONFIG['dry_run'],
)

started_at = time.perf_counter()
result = run_baseline_or_raise(cfg)
elapsed_seconds = round(time.perf_counter() - started_at, 2)
{'elapsed_seconds': elapsed_seconds, 'result': result.to_dict()}


## 7. Post-Run Artifacts


In [ ]:
from pathlib import Path

from training.lib.qlora_smoke import (
    load_json_artifact,
    load_optional_json_artifact,
    load_sample_generations,
    summarize_sample_generations,
)

run_config_artifact = load_json_artifact(result.output_dir, 'run_config.json')
summary_artifact = load_json_artifact(result.output_dir, 'summary.json')
metrics_artifact = load_optional_json_artifact(result.output_dir, 'metrics.json')
sample_rows = load_sample_generations(result.output_dir)
sample_summary = summarize_sample_generations(sample_rows) if sample_rows else None
artifact_preview = {
    'status': summary_artifact.get('status'),
    'used_true_qlora': summary_artifact.get('used_true_qlora'),
    'runtime': summary_artifact.get('runtime'),
    'train_loss': summary_artifact.get('train_loss'),
    'eval_loss': summary_artifact.get('eval_loss'),
    'output_dir': summary_artifact.get('output_dir'),
    'adapter_dir': summary_artifact.get('adapter_dir'),
    'sample_count': len(sample_rows),
    'sample_summary': sample_summary,
    'run_config_artifact': run_config_artifact,
    'summary_artifact': summary_artifact,
    'metrics_artifact': metrics_artifact,
}
artifact_preview


## 8. Training Metrics


In [ ]:
import json

import matplotlib.pyplot as plt

trainer_state_path = Path(result.output_dir) / 'checkpoints' / 'trainer_state.json'
trainer_state = None
if trainer_state_path.exists():
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8'))

if trainer_state and trainer_state.get('log_history'):
    train_points = [(entry['step'], entry['loss']) for entry in trainer_state['log_history'] if 'loss' in entry]
    eval_points = [(entry['step'], entry['eval_loss']) for entry in trainer_state['log_history'] if 'eval_loss' in entry]
    plt.figure(figsize=(8, 4))
    if train_points:
        plt.plot([step for step, _ in train_points], [loss for _, loss in train_points], label='train_loss')
    if eval_points:
        plt.plot([step for step, _ in eval_points], [loss for _, loss in eval_points], label='eval_loss')
    plt.xlabel('step')
    plt.ylabel('loss')
    plt.title('WorldSim baseline training losses')
    plt.legend()
    plt.show()
else:
    final_metrics = {
        'train_loss': summary_artifact.get('train_loss'),
        'eval_loss': summary_artifact.get('eval_loss'),
    }
    plt.figure(figsize=(5, 4))
    plt.bar(final_metrics.keys(), [value if value is not None else 0.0 for value in final_metrics.values()])
    plt.title('Final losses (no trainer_state log history found)')
    plt.show()
    final_metrics


## 9. Analyzer + Registry + Candidate Selection


In [ ]:
import json

from tools.generation_analyzer import generate_report, recommend_next_action
from training.lib.qlora_smoke import (
    BEST_ADAPTER_POINTER_PATH,
    MODEL_REGISTRY_PATH,
    build_baseline_candidate_judgment,
    load_model_registry,
    register_baseline_run,
    select_best_adapter_run,
    update_best_adapter_pointer,
)

analysis_report = generate_report(sample_rows, examples_per_category=3) if sample_rows else None
analysis_report_path = Path(result.output_dir) / 'analysis_report.json'
if analysis_report is not None:
    analysis_report_path.write_text(json.dumps(analysis_report, ensure_ascii=False, indent=2), encoding='utf-8')
analysis_recommendation = recommend_next_action(analysis_report) if analysis_report else None
retry_samples = sum(1 for row in sample_rows if int(row.get('structured_attempt_count', 1) or 1) > 1)
retry_rate = round(retry_samples / len(sample_rows), 4) if sample_rows else None
registry_entry = register_baseline_run(
    MODEL_REGISTRY_PATH,
    config={**CONFIG, 'analysis_report_path': str(analysis_report_path) if analysis_report is not None else None},
    result=result,
    analysis_report=analysis_report,
    metrics={'retry_rate': retry_rate},
    created_at=run_config_artifact.get('generated_at'),
)
registry = load_model_registry(MODEL_REGISTRY_PATH)
best_run = select_best_adapter_run(registry)
best_adapter = update_best_adapter_pointer(BEST_ADAPTER_POINTER_PATH, best_run)
baseline_judgment = build_baseline_candidate_judgment(result, analysis_report)
{
    'analysis_report_path': str(analysis_report_path) if analysis_report is not None else None,
    'analysis_report': analysis_report,
    'analysis_recommendation': analysis_recommendation,
    'retry_rate': retry_rate,
    'registry_path': str(MODEL_REGISTRY_PATH),
    'registry_entry': registry_entry,
    'registered_runs': len(registry.get('runs', [])),
    'best_run': best_run,
    'best_adapter_pointer': str(BEST_ADAPTER_POINTER_PATH),
    'best_adapter': best_adapter,
    'baseline_judgment': baseline_judgment,
}


## 10. Final Operational Judgment


In [ ]:
final_operational_judgment = {
    **baseline_judgment,
    'malformed_json': analysis_report.get('malformed_json_count') if analysis_report else None,
    'fenced_json': analysis_report.get('fenced_json_count') if analysis_report else None,
    'enum_drift_total': sample_summary.get('enum_drift_total') if sample_summary else None,
    'semantic_valid': sample_summary.get('semantic_valid') if sample_summary else None,
    'semantic_low_quality': analysis_report.get('semantic_low_quality_count') if analysis_report else None,
    'semantic_drift': analysis_report.get('semantic_drift_count') if analysis_report else None,
    'language_drift': analysis_report.get('language_drift_count') if analysis_report else None,
    'analyzer_overall_status': analysis_report.get('overall_status') if analysis_report else None,
}
final_operational_judgment
